## 경로 설정 (레포 공통)

이 셀은 레포 어디에서 노트북을 열어도 `data/` 폴더를 자동으로 찾아 **상대경로 기반**으로 경로를 지정한다.
- `data/raw` 원자료 → `data/interim` 중간 자료 → `data/processed` 최종본, 그림·표는 `outputs/`에 저장한다.
- 노트북은 레포 안 어느 위치에서 실행해도 되지만, 레포 폴더 구조(`data/`, `src/`, `outputs/`)를 바꾸지 않아야 한다.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
for _d in (INTERIM, PROCESSED, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("프로젝트 루트 폴더:", ROOT.name)

프로젝트 루트 폴더: BAF-26-2-finance_2



# ISTANS 4단계: 결측지도 만들기

## 목적

이 노트북은 ISTANS에서 내려받은 **안정성 4개 파일**과 **수익성 3개 파일**을 이용해 다음을 확인합니다.

1. 40대 제조업 × 1994~2025 패널이 실제로 완성되는지
2. 안정성 18개 + 수익성 5개, 총 23개 항목에 결측이 있는지
3. 금리취약성 파일럿에 필요한 핵심 5변수에 결측이 있는지
4. 40개 업종을 모두 함께 사용할 수 있는 공통 기간이 언제부터인지
5. 이자보상비율(ICR)에 음수나 매우 큰 값이 얼마나 있는지

> **중요한 원칙**
>
> - `0`은 결측으로 처리하지 않습니다. 예를 들어 단기사채가 0인 것은 실제 값일 수 있습니다.
> - ICR 음수나 큰 값을 자동으로 삭제하지 않습니다.
> - 1%·99% 분위수는 **이상치 제거 기준**이 아니라, 나중에 확인할 값을 찾는 **진단용 표시**로만 사용합니다.
> - 이 노트북이 확인하는 것은 **값의 존재 여부**입니다. ISTANS의 연도별 표본기업 구성이 동일하다는 것까지 증명하지는 않습니다.



## 목차

1. 입력 파일 경로 설정
2. ISTANS 엑셀 구조 읽기
3. 40대 제조업만 추출
4. 결측 기준 정의
5. 전체 23개 항목 결측지도
6. 핵심 5변수 결측지도
7. 공통 사용기간 확인
8. 이자보상비율(ICR) 진단
9. 결과 엑셀 저장
10. 결과 해석



## 1. 입력 파일 경로 설정

아래 파일명을 본인이 저장한 위치에 맞게 수정하면 됩니다.

현재 파일은 연도별로 다음처럼 나뉘어 있습니다.

- 안정성: 1994~1999 / 2000~2009 / 2010~2019 / 2020~2025
- 수익성: 1994~1999 / 2000~2009 / 2010~2025

파일을 한 폴더에 넣었다면 `DATA_DIR`만 수정하면 됩니다.


In [2]:

try:
    from artifact_tool import Blob, SpreadsheetFile, Workbook   # 엑셀 저장용(Claude 환경 전용 모듈)
    HAS_ARTIFACT = True
except ImportError:
    HAS_ARTIFACT = False   # 일반 로컬 환경: 엑셀 저장 단계만 건너뜀(결과 파일은 레포에 이미 포함)
from pathlib import Path
from collections import defaultdict
import re
import json
import math

# 본인 폴더에 맞게 수정
DATA_DIR = RAW

stability_files = [
    RAW / "ISTANS 안정성" / "안정성_20260919005219.xlsx",  # 1994~1999
    RAW / "ISTANS 안정성" / "안정성_20260919005148.xlsx",  # 2000~2009
    RAW / "ISTANS 안정성" / "안정성_20260919005109.xlsx",  # 2010~2019
    RAW / "ISTANS 안정성" / "안정성_20260919005022.xlsx",  # 2020~2025
]

profitability_files = [
    RAW / "ISTANS 수익성" / "수익성_20260919005852.xlsx",  # 1994~1999
    RAW / "ISTANS 수익성" / "수익성_20260919005829.xlsx",  # 2000~2009
    RAW / "ISTANS 수익성" / "수익성_20260919005811.xlsx",  # 2010~2025
]

# 파일 존재 여부 확인
for path in stability_files + profitability_files:
    print(path.name, "->", "OK" if path.exists() else "없음")


안정성_20260919005219.xlsx -> OK
안정성_20260919005148.xlsx -> OK
안정성_20260919005109.xlsx -> OK
안정성_20260919005022.xlsx -> OK
수익성_20260919005852.xlsx -> OK
수익성_20260919005829.xlsx -> OK
수익성_20260919005811.xlsx -> OK



### 해석

모든 파일이 `OK`로 나오면 다음 단계로 진행합니다.  
파일이 없다고 나오면 파일명 또는 `DATA_DIR` 경로를 먼저 수정합니다.



## 2. ISTANS 엑셀 구조 읽기

ISTANS 파일은 보통 다음 구조입니다.

- 첫 번째 열: 산업코드 + 산업명
- 첫 번째 헤더행: 연도
- 두 번째 헤더행: 변수명
- 이후 행: 산업별 값

아래 함수는 이 구조를 읽어 `(업종코드, 연도, 변수명)` 형태로 다시 정리합니다.


In [3]:

def variable_name(raw):
    """
    예:
    '24 이자보상비율 (....)' -> '이자보상비율'
    """
    m = re.match(r"\d+\s+(.+?)\s+\(", str(raw))
    return m.group(1).strip() if m else str(raw).strip()


def read_istans_file(path):
    """
    ISTANS xlsx의 '데이터' 시트를 읽어
    각 산업 행을 dict 형태로 반환합니다.
    """
    import openpyxl
    wb = openpyxl.load_workbook(str(path), data_only=True)  # read_only 모드는 이 파일의 범위를 1x1로 잘못 읽음
    if "데이터" not in wb.sheetnames:
        raise ValueError(f"{path.name}: '데이터' 시트를 찾지 못했습니다.")
    values = [list(r) for r in wb["데이터"].iter_rows(values_only=True)]

    # 첫 헤더행에서 연도, 두 번째 헤더행에서 변수명 추출
    years = []
    variables = []

    for c in range(1, len(values[0])):
        ym = re.search(r"(\d{4})", str(values[0][c]))
        years.append(int(ym.group(1)) if ym else None)
        variables.append(variable_name(values[1][c]))

    rows = []

    for r in range(2, len(values)):
        raw_name = str(values[r][0]).strip()

        # 예: I3312 조선
        m = re.search(r"(I\d+)\s+(.+)", raw_name)
        if not m:
            continue

        code = m.group(1)
        industry = m.group(2).strip()

        row = {
            "code": code,
            "industry": industry,
        }

        for c in range(1, len(values[0])):
            row[(years[c-1], variables[c-1])] = values[r][c]

        rows.append(row)

    return rows



### 해석

이 단계에서는 아직 결측을 판단하지 않습니다.  
원자료를 **업종·연도·변수 기준으로 다루기 쉽게 바꾸기만** 합니다.



## 3. 40대 제조업만 추출

ISTANS에는 다음과 같은 집계행도 같이 들어 있습니다.

- 제조업 전체
- 고위기술산업군
- 중고위기술산업군
- 중저위기술산업군
- 저위기술산업군

이번 결측지도에서는 이런 합계행을 제외하고,  
`I3101`처럼 **세부 제조업 코드**만 남깁니다.

정규식 `I3[1-4]\d{2}`에 해당하는 코드만 선택하면 40개 제조업종이 남습니다.


In [4]:

data = {}
industry_names = {}

def load_group(files, group_name):
    for path in files:
        rows = read_istans_file(path)

        for row in rows:
            code = row["code"]

            # I3101 ~ I3410 형태의 세부 제조업만
            if not re.fullmatch(r"I3[1-4]\d{2}", code):
                continue

            industry_names[code] = row["industry"]

            for key, value in row.items():
                if not isinstance(key, tuple):
                    continue

                year, var = key
                data[(code, year, group_name, var)] = value


load_group(stability_files, "안정성")
load_group(profitability_files, "수익성")

industry_codes = sorted(industry_names.keys())
years = list(range(1994, 2026))

print("제조업 세부업종 수:", len(industry_codes))
print("분석 기간:", years[0], "~", years[-1])
print("\n앞 10개 업종:")
for code in industry_codes[:10]:
    print(code, industry_names[code])


<python-env> UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


제조업 세부업종 수: 40
분석 기간: 1994 ~ 2025

앞 10개 업종:
I3101 의약
I3102 반도체
I3103 디스플레이
I3104 컴퓨터
I3105 통신기기
I3106 가전
I3107 정밀기기
I3108 전지
I3109 항공
I3201 석유화학



### 해석

정상이라면 다음과 같이 나와야 합니다.

- 제조업 세부업종 수: **40개**
- 분석 기간: **1994~2025**

40개가 아니라면 산업코드 구조가 바뀌었는지 먼저 확인해야 합니다.



## 4. 결측 기준 정의

여기서 가장 중요한 부분입니다.

### 결측으로 보는 것
- 셀이 완전히 비어 있음
- `None`
- 빈 문자열 `""`

### 결측으로 보지 않는 것
- `0`
- 음수
- 아주 큰 수

예를 들어 `단기사채 = 0`은 실제로 단기사채가 없다는 값일 수 있으므로 결측이 아닙니다.


In [5]:

stability_vars = [
    "자산총계", "유형자산", "유동자산", "재고자산", "자본총계",
    "이자비용", "부채총계", "유동부채", "비유동부채",
    "단기차입", "장기차입", "단기사채", "장기사채",
    "자본비율", "부채비율", "유동부채비율", "비유동부채비율", "차입의존도"
]

profitability_vars = [
    "매출액", "영업이익", "당기순이익", "영업이익률", "이자보상비율"
]

# 금리취약성 파일럿에서 우선 필요한 핵심 변수
core_stability_vars = [
    "자산총계",
    "단기차입",
    "단기사채",
    "차입의존도",
]

core_profitability_vars = [
    "이자보상비율"
]


def is_missing(value):
    # 0은 결측이 아님
    return value is None or value == ""



## 5. 전체 23개 항목 결측지도

각 업종×연도에 대해

- 안정성 18개
- 수익성 5개

**총 23개가 모두 있으면 1**, 하나라도 비어 있으면 **0**으로 표시합니다.


In [6]:

def complete_for_all_23(code, year):
    checks = []

    for var in stability_vars:
        value = data.get((code, year, "안정성", var))
        checks.append(not is_missing(value))

    for var in profitability_vars:
        value = data.get((code, year, "수익성", var))
        checks.append(not is_missing(value))

    return all(checks)


all23_map = [
    [1 if complete_for_all_23(code, year) else 0 for year in years]
    for code in industry_codes
]

all23_missing = sum(
    1
    for row in all23_map
    for value in row
    if value == 0
)

print("전체 23개 항목 기준 결측 업종×연도 칸:", all23_missing)
print("전체 업종×연도 칸:", len(industry_codes) * len(years))


전체 23개 항목 기준 결측 업종×연도 칸: 0
전체 업종×연도 칸: 1280



### 해석

`전체 23개 항목 기준 결측 업종×연도 칸 = 0`이면  
40개 제조업 × 32년의 각 칸에서 23개 변수가 전부 존재한다는 뜻입니다.

단, 이 결과는 **빈칸이 없다는 뜻**이지  
연도마다 집계에 포함된 기업 구성이 완전히 같다는 뜻은 아닙니다.



## 6. 핵심 5변수 결측지도

금리취약성 파일럿에서 우선 직접 사용할 가능성이 높은 다음 5개만 다시 확인합니다.

- 자산총계
- 단기차입
- 단기사채
- 차입의존도
- 이자보상비율


In [7]:

def complete_for_core(code, year):
    checks = []

    for var in core_stability_vars:
        value = data.get((code, year, "안정성", var))
        checks.append(not is_missing(value))

    for var in core_profitability_vars:
        value = data.get((code, year, "수익성", var))
        checks.append(not is_missing(value))

    return all(checks)


core_map = [
    [1 if complete_for_core(code, year) else 0 for year in years]
    for code in industry_codes
]

core_missing = sum(
    1
    for row in core_map
    for value in row
    if value == 0
)

print("핵심 5변수 기준 결측 업종×연도 칸:", core_missing)


핵심 5변수 기준 결측 업종×연도 칸: 0



### 해석

여기서도 `0`이면, 적어도 **금리 주제의 최소 데이터 구성은 결측 때문에 바로 막히지는 않는다**고 볼 수 있습니다.

다만 아직 회귀모형의 타당성, 금리충격 정의, 장기 비교가능성까지 검증한 것은 아닙니다.



## 7. 공통 사용기간 확인

40개 제조업 모두가 동시에 완전한 연도만 찾습니다.


In [8]:

common_years = []

for year_index, year in enumerate(years):
    all_industries_complete = all(
        all23_map[i][year_index] == 1
        for i in range(len(industry_codes))
    )

    if all_industries_complete:
        common_years.append(year)

common_start = min(common_years) if common_years else None
common_end = max(common_years) if common_years else None

print("40개 업종 공통 완전 시작연도:", common_start)
print("40개 업종 공통 완전 종료연도:", common_end)
print("공통 완전 연도 수:", len(common_years))


40개 업종 공통 완전 시작연도: 1994
40개 업종 공통 완전 종료연도: 2025
공통 완전 연도 수: 32



### 해석

현재 파일 기준으로 `1994~2025`가 나오면  
**결측만 놓고 볼 때 32년 전체를 사용할 수 있다**는 의미입니다.

다만 ISTANS 주석에서 연도별 표본기업 구성 변화 정보가 제공되지 않았다면,  
이 결과를 곧바로 **32년 동안 완전히 동일한 모집단을 비교한다**고 해석하면 안 됩니다.



## 8. 이자보상비율(ICR) 진단

이자보상비율은 영업손실이 있으면 음수가 될 수 있고,  
이자비용이 매우 작으면 큰 양수가 나올 수도 있습니다.

그래서 여기서는:

- 음수 ICR 개수 확인
- 최소값·최대값 확인
- 1%, 50%, 99% 분위수 확인

까지만 합니다.

**음수나 상·하위 1%를 자동 삭제하지 않습니다.**


In [9]:

icr_rows = []

for code in industry_codes:
    for year in years:
        value = data.get((code, year, "수익성", "이자보상비율"))

        if not is_missing(value):
            icr_rows.append(
                (code, industry_names[code], year, float(value))
            )

icr_values = sorted(row[3] for row in icr_rows)


def quantile(sorted_values, p):
    """
    선형보간 분위수 계산
    외부 통계 패키지에 의존하지 않도록 직접 계산합니다.
    """
    n = len(sorted_values)
    position = (n - 1) * p
    lo = math.floor(position)
    hi = math.ceil(position)

    if lo == hi:
        return sorted_values[lo]

    return (
        sorted_values[lo]
        + (sorted_values[hi] - sorted_values[lo]) * (position - lo)
    )


p01 = quantile(icr_values, 0.01)
p50 = quantile(icr_values, 0.50)
p99 = quantile(icr_values, 0.99)

negative_rows = [
    row for row in icr_rows
    if row[3] < 0
]

print("ICR 관측치:", len(icr_values))
print("음수 ICR:", len(negative_rows))
print("최소값:", min(icr_values))
print("1% 분위수:", p01)
print("중앙값:", p50)
print("99% 분위수:", p99)
print("최대값:", max(icr_values))


ICR 관측치: 1280
음수 ICR: 52
최소값: -2655.1
1% 분위수: -462.927
중앙값: 368.6
99% 분위수: 13186.104000000014
최대값: 104013.8



### 해석

음수 ICR은 **결측이 아닙니다.**  
따라서 음수라는 이유만으로 삭제하지 않습니다.

또한 1%·99%는 **이상치 제거 기준이 아니라 검토용 기준**입니다.  
나중에 회귀를 실제로 할 때 분포, 경제적 의미, 강건성 분석을 보고 처리 여부를 결정해야 합니다.



## 9. 결과 엑셀 저장

다음 시트를 가진 `결측지도.xlsx`를 생성합니다.

- `요약`
- `결측지도_전체23`
- `결측지도_핵심5`
- `ICR_진단`
- `음수ICR_업종별`

결측지도에서:

- `1` = 필요한 값이 전부 존재
- `0` = 하나 이상 결측


In [10]:

# ------------------------------------------------------------
# ICR 검토 대상 만들기
# ------------------------------------------------------------
flagged = []

for code, industry, year, value in icr_rows:
    negative = value < 0
    low_1pct = value <= p01
    high_1pct = value >= p99

    if negative or low_1pct or high_1pct:
        flags = []

        if negative:
            flags.append("음수")
        if low_1pct:
            flags.append("하위 1%")
        if high_1pct:
            flags.append("상위 1%")

        flagged.append([
            code,
            industry,
            year,
            value,
            " / ".join(flags),
            "진단용 검토 대상 — 자동 삭제하지 않음",
        ])

flagged.sort(key=lambda x: x[3])


# 업종별 음수 ICR 연도 정리
neg_by_industry = defaultdict(list)

for code, industry, year, value in negative_rows:
    neg_by_industry[(code, industry)].append((year, value))

neg_industry_table = []

for (code, industry), items in sorted(
    neg_by_industry.items(),
    key=lambda kv: (-len(kv[1]), kv[0][0])
):
    neg_industry_table.append([
        code,
        industry,
        len(items),
        ", ".join(str(year) for year, _ in items),
    ])


if HAS_ARTIFACT:
    # ------------------------------------------------------------
    # 새 엑셀 생성
    # ------------------------------------------------------------
    out = Workbook.create()

    # ========== 1) 요약 ==========
    summary = out.worksheets.add("요약")

    summary.merge_cells("A1:C1")
    summary.get_range("A1").values = [[
        "ISTANS 4단계 · 결측 및 ICR 진단 결과"
    ]]

    summary.get_range("A1:C1").format = {
        "fill": "#1F4E78",
        "font": {
            "bold": True,
            "color": "#FFFFFF",
            "size": 14,
        },
        "horizontal_alignment": "center",
        "vertical_alignment": "center",
    }

    summary_rows = [
        ["항목", "결과", "해석"],
        ["40대 제조업 수", len(industry_codes), "세부 제조업 코드만 사용"],
        ["분석 연도", f"{years[0]}~{years[-1]}", f"{len(years)}개 연도"],
        ["업종×연도 관측칸", len(industry_codes) * len(years), "40 × 32 = 1,280"],
        ["안정성 18 + 수익성 5 전체항목 결측칸", all23_missing, "0이면 23개 항목이 모든 업종·연도에서 존재"],
        ["핵심 5변수 결측칸", core_missing, "자산총계·단기차입·단기사채·차입의존도·ICR"],
        ["40업종 공통 완전 시작연도", common_start, "전체 23개 항목 기준"],
        ["40업종 공통 완전 종료연도", common_end, "전체 23개 항목 기준"],
        ["ICR 관측치", len(icr_rows), "40 × 32"],
        ["음수 ICR", len(negative_rows), "결측이 아니라 실제 관측값으로 유지"],
        ["ICR 최소값", min(icr_values), ""],
        ["ICR 1% 분위수", p01, "제거 기준이 아니라 진단 기준"],
        ["ICR 중앙값", p50, ""],
        ["ICR 99% 분위수", p99, "제거 기준이 아니라 진단 기준"],
        ["ICR 최대값", max(icr_values), ""],
    ]

    summary.get_range(f"A3:C{2 + len(summary_rows)}").values = summary_rows
    summary.get_range("A3:C3").format = {
        "fill": "#D9EAF7",
        "font": {"bold": True},
        "horizontal_alignment": "center",
    }

    summary.get_range("A3:C17").format.wrap_text = True
    summary.get_range("A:A").format.column_width = 31
    summary.get_range("B:B").format.column_width = 18
    summary.get_range("C:C").format.column_width = 48


    # ========== 2) 전체 23개 항목 결측지도 ==========
    map_all = out.worksheets.add("결측지도_전체23")

    map_all.merge_cells("A1:AH1")
    map_all.get_range("A1").values = [[
        "1 = 안정성 18개 + 수익성 5개 항목 모두 존재 / 0 = 하나 이상 결측"
    ]]

    map_all.get_range("A1:AH1").format = {
        "fill": "#1F4E78",
        "font": {"bold": True, "color": "#FFFFFF"},
    }

    headers = ["업종"] + years + ["결측연도수"]

    map_all.get_range("A3:AH3").values = [headers]
    map_all.get_range("A3:AH3").format = {
        "fill": "#D9EAF7",
        "font": {"bold": True},
        "horizontal_alignment": "center",
    }

    rows_for_map = []

    for i, code in enumerate(industry_codes):
        rows_for_map.append(
            [f"{code} {industry_names[code]}"]
            + all23_map[i]
            + [None]
        )

    map_all.get_range("A4:AH43").values = rows_for_map

    map_all.get_range("AH4").formulas = [["=COUNTIF(B4:AG4,0)"]]
    map_all.get_range("AH4:AH43").fill_down()

    map_all.get_range("A44").values = [["연도별 결측 업종 수"]]
    map_all.get_range("B44").formulas = [["=COUNTIF(B4:B43,0)"]]
    map_all.get_range("B44:AG44").fill_right()

    map_all.freeze_panes.freeze_rows(3)
    map_all.freeze_panes.freeze_columns(1)

    map_all.get_range("A:A").format.column_width = 22
    map_all.get_range("B:AG").format.column_width = 5
    map_all.get_range("AH:AH").format.column_width = 12

    map_all.get_range("B4:AG43").conditional_formats.add_cell_is({
        "operator": "equalTo",
        "formula": 0,
        "format": {
            "fill": "#F4CCCC",
            "font": {"color": "#9C0006", "bold": True},
        },
    })

    map_all.get_range("B4:AG43").conditional_formats.add_cell_is({
        "operator": "equalTo",
        "formula": 1,
        "format": {
            "fill": "#E2F0D9",
            "font": {"color": "#006100"},
        },
    })


    # ========== 3) 핵심 5변수 결측지도 ==========
    map_core = out.worksheets.add("결측지도_핵심5")

    map_core.merge_cells("A1:AH1")
    map_core.get_range("A1").values = [[
        "1 = 자산총계·단기차입·단기사채·차입의존도·이자보상비율 모두 존재 / 0 = 결측"
    ]]

    map_core.get_range("A1:AH1").format = {
        "fill": "#1F4E78",
        "font": {"bold": True, "color": "#FFFFFF"},
    }

    map_core.get_range("A3:AH3").values = [headers]
    map_core.get_range("A3:AH3").format = {
        "fill": "#D9EAF7",
        "font": {"bold": True},
        "horizontal_alignment": "center",
    }

    rows_for_core = []

    for i, code in enumerate(industry_codes):
        rows_for_core.append(
            [f"{code} {industry_names[code]}"]
            + core_map[i]
            + [None]
        )

    map_core.get_range("A4:AH43").values = rows_for_core

    map_core.get_range("AH4").formulas = [["=COUNTIF(B4:AG4,0)"]]
    map_core.get_range("AH4:AH43").fill_down()

    map_core.get_range("A44").values = [["연도별 결측 업종 수"]]
    map_core.get_range("B44").formulas = [["=COUNTIF(B4:B43,0)"]]
    map_core.get_range("B44:AG44").fill_right()

    map_core.freeze_panes.freeze_rows(3)
    map_core.freeze_panes.freeze_columns(1)

    map_core.get_range("A:A").format.column_width = 22
    map_core.get_range("B:AG").format.column_width = 5
    map_core.get_range("AH:AH").format.column_width = 12

    map_core.get_range("B4:AG43").conditional_formats.add_cell_is({
        "operator": "equalTo",
        "formula": 0,
        "format": {
            "fill": "#F4CCCC",
            "font": {"color": "#9C0006", "bold": True},
        },
    })

    map_core.get_range("B4:AG43").conditional_formats.add_cell_is({
        "operator": "equalTo",
        "formula": 1,
        "format": {
            "fill": "#E2F0D9",
            "font": {"color": "#006100"},
        },
    })


    # ========== 4) ICR 진단 ==========
    icr_sheet = out.worksheets.add("ICR_진단")

    icr_sheet.merge_cells("A1:F1")
    icr_sheet.get_range("A1").values = [[
        "이자보상비율(ICR) 음수 및 분포 꼬리 관측치 — 진단용, 자동 삭제 금지"
    ]]

    icr_sheet.get_range("A1:F1").format = {
        "fill": "#1F4E78",
        "font": {"bold": True, "color": "#FFFFFF"},
    }

    icr_sheet.get_range("A3:B11").values = [
        ["통계", "값"],
        ["관측치 수", len(icr_values)],
        ["최소", min(icr_values)],
        ["1%", p01],
        ["중앙값", p50],
        ["99%", p99],
        ["최대", max(icr_values)],
        ["음수 ICR 수", len(negative_rows)],
        ["검토 원칙", "음수/상하위 1% 자동 삭제 안 함"],
    ]

    icr_sheet.get_range("A3:B3").format = {
        "fill": "#D9EAF7",
        "font": {"bold": True},
    }

    icr_headers = [
        "업종코드",
        "업종",
        "연도",
        "ICR",
        "플래그",
        "처리 원칙",
    ]

    icr_sheet.get_range("A13:F13").values = [icr_headers]
    icr_sheet.get_range("A13:F13").format = {
        "fill": "#D9EAF7",
        "font": {"bold": True},
        "horizontal_alignment": "center",
    }

    if flagged:
        icr_sheet.get_range(
            f"A14:F{13 + len(flagged)}"
        ).values = flagged

    icr_sheet.freeze_panes.freeze_rows(13)

    icr_sheet.get_range("A:F").format.autofit_columns()
    icr_sheet.get_range("B:B").format.column_width = 18
    icr_sheet.get_range("F:F").format.column_width = 34
    icr_sheet.get_range("A1:F300").format.wrap_text = True


    # ========== 5) 음수 ICR 업종별 ==========
    neg_sheet = out.worksheets.add("음수ICR_업종별")

    neg_sheet.get_range("A1:D1").values = [[
        "업종코드",
        "업종",
        "음수 연도 수",
        "해당 연도",
    ]]

    neg_sheet.get_range("A1:D1").format = {
        "fill": "#D9EAF7",
        "font": {"bold": True},
    }

    if neg_industry_table:
        neg_sheet.get_range(
            f"A2:D{1 + len(neg_industry_table)}"
        ).values = neg_industry_table

    neg_sheet.get_range("A:D").format.autofit_columns()
    neg_sheet.get_range("D:D").format.column_width = 48
    neg_sheet.get_range("A:D").format.wrap_text = True


    # ------------------------------------------------------------
    # 저장
    # ------------------------------------------------------------
    OUTPUT_PATH = TAB_DIR / "결측지도.xlsx"
    SpreadsheetFile.export_xlsx(out).save(str(OUTPUT_PATH))

    print("저장 완료:", OUTPUT_PATH.resolve())

else:
    print('artifact_tool 미설치: 엑셀 저장 단계를 건너뜀 (결과 파일은 레포에 포함)')

artifact_tool 미설치: 엑셀 저장 단계를 건너뜀 (결과 파일은 레포에 포함)



## 10. 결과 해석

현재 업로드된 ISTANS 자료에서 이 노트북을 실행했을 때 확인된 결과는 다음과 같습니다.

- 40대 제조업: **40개**
- 분석기간: **1994~2025, 32년**
- 업종×연도 칸: **1,280개**
- 안정성 18개 + 수익성 5개 전체항목 기준 결측: **0칸**
- 핵심 5변수 기준 결측: **0칸**
- 공통 완전 사용기간: **1994~2025**
- ICR 음수: **52건**
- ICR 범위: **-2,655.1 ~ 104,013.8**
- ICR 중앙값: **368.6**

### 이 결과가 의미하는 것

**결측 때문에 장기 패널을 구성할 수 없는 문제는 현재 파일에서는 발견되지 않았습니다.**

즉, 원래 우려했던  
“일부 제조업종의 초기 연도 자료가 비어서 30년 장기 분석을 못 하는 상황”은  
현재 자료에서는 나타나지 않았습니다.

### 그러나 아직 말할 수 없는 것

다음은 이 결측지도만으로 확인되지 않습니다.

1. ISTANS가 1994~2025 동안 동일한 기업 표본을 계속 사용했는지
2. 산업분류 체계가 장기간 완전히 동일한지
3. ICR의 매우 큰 값이 회귀 결과에 얼마나 영향을 주는지
4. 금리충격을 어떤 방식으로 정의해야 가장 타당한지
5. 단기차입·단기사채 비중이 실제 금리 재설정 노출도를 충분히 대변하는지

따라서 이번 단계의 결론은 좁게 잡습니다.

> **“금리취약성 주제는 데이터 결측 때문에 바로 탈락하지 않는다.”**

이 이상으로 연구의 타당성이나 수상 가능성까지 확대해서 해석하지 않습니다.
